# Workshop 8: NLP - Text Pre-processing and Text Representations

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Part 1: Text Pre-processing in NLP

### Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('punkt')      # Required for word_tokenize
nltk.download('wordnet')    # Required for lemmatization
nltk.download('omw-1.4')


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
# Loading the dataset for pre-processing
# Using 'trumptweets_small.csv' as it is available in the directory
df = pd.read_csv('/content/drive/MyDrive/Machine Learning/Week8/trumptweets_small.csv')
df_text = df[['content']].dropna()
print(f"Loaded {len(df_text)} tweets for pre-processing.")
df_text.head()

Loaded 41122 tweets for pre-processing.


,content
0,Be sure to tune in and watch Donald Trump on L...
1,Donald Trump will be appearing on The View tom...
2,Donald Trump reads Top Ten Financial Tips on L...
3,New Blog Post: Celebrity Apprentice Finale and...
4,"""My persona will never be that of a wallflower..."


### Defining Cleaning Functions

In [ ]:
def remove_urls(text):
    """Removes URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def remove_emoji(text):
    """Removes emojis from the text."""
    emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"
                               u"\U0001F300-\U0001F5FF"
                               u"\U0001F680-\U0001F6FF"
                               u"\U0001F1E0-\U0001F1FF"
                               u"\U00002702-\U000027B0"
                               u"\U000024C2-\U0001F251"
                               "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r' ', text)

def removeunwanted_characters(document):
    """Removes mentions, hashtags, punctuation, and extra whitespace."""
    # remove user mentions
    document = re.sub("@[A-Za-z0-9_]+", " ", document)
    # remove hashtags
    document = re.sub("#[A-Za-z0-9_]+", "", document)
    # remove punctuation and non-alphanumeric characters (keep spaces)
    document = re.sub("[^0-9A-Za-z ]", "", document)
    # remove emojis
    document = remove_emoji(document)
    # remove double spaces
    document = re.sub(r'\s+', ' ', document)
    return document.strip()

In [ ]:
def remove_stopwords(text_tokens):
    """Removes stopwords from a list of tokens."""
    stop_words = set(stopwords.words('english'))
    custom_stopwords = ['@', 'RT']
    stop_words.update(custom_stopwords)
    return [token for token in text_tokens if token not in stop_words]

def lemmatization(token_text):
    """Performs lemmatization on a list of tokens."""
    wordnet = WordNetLemmatizer()
    return [wordnet.lemmatize(token, pos='v') for token in token_text]

def stemming(token_text):
    """Performs stemming on a list of tokens."""
    ps = PorterStemmer()
    return [ps.stem(token) for token in token_text]

### Testing Pre-processing Components

In [ ]:
import nltk

# Simplified test string to avoid Unicode surrogate encoding issues in some environments
test_string = "Hello @siman, still up for the movie??? https://movies.com #MovieNight #friday"
print(f"Original: {test_string}")

no_url = remove_urls(test_string)
cleaned = removeunwanted_characters(no_url)
print(f"Cleaned: {cleaned}")

tokens = word_tokenize(cleaned.lower())
filtered_tokens = remove_stopwords(tokens)
lemmatized = lemmatization(filtered_tokens)
stemmed = stemming(filtered_tokens)

print(f"Tokens: {tokens}")
print(f"Filtered: {filtered_tokens}")
print(f"Lemmatized: {lemmatized}")
print(f"Stemmed: {stemmed}")

Original: Hello @siman, still up for the movie??? https://movies.com #MovieNight #friday
Cleaned: Hello still up for the movie
Tokens: ['hello', 'still', 'up', 'for', 'the', 'movie']
Filtered: ['hello', 'still', 'movie']
Lemmatized: ['hello', 'still', 'movie']
Stemmed: ['hello', 'still', 'movi']


## Part 2: Text Classification Pipeline

### Building the Pipeline Function

In [ ]:
def text_cleaning_pipeline(text, rule="lemmatize"):
    """
    Full pipeline for cleaning and normalizing text.
    """
    if not isinstance(text, str):
        return ""

    # 1. Lowercase
    text = text.lower()

    # 2. Remove URLs
    text = remove_urls(text)

    # 3. Remove Emojis and Unwanted Characters
    text = removeunwanted_characters(text)

    # 4. Tokenization
    tokens = word_tokenize(text)

    # 5. Remove Stopwords
    tokens = remove_stopwords(tokens)

    # 6. Normalization (Stemming or Lemmatization)
    if rule == "lemmatize":
        tokens = lemmatization(tokens)
    elif rule == "stem":
        tokens = stemming(tokens)

    return " ".join(tokens)

### Sentiment Classification Task

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 1. Load the Dataset
try:
    # Note: Filename corrected to match disk content
    df_sentiment = pd.read_csv('/content/drive/MyDrive/Machine Learning/Week8/trum_tweet_sentiment_analysis.csv')

    # Inspect columns to handle flexible naming
    print("Columns found:", df_sentiment.columns.tolist())

    # Map columns based on inspection (text, Sentiment)
    text_col = 'text' if 'text' in df_sentiment.columns else df_sentiment.columns[0]
    label_col = 'Sentiment' if 'Sentiment' in df_sentiment.columns else 'label'

    if label_col not in df_sentiment.columns:
        label_col = df_sentiment.columns[1] # fallback to second column

    print(f"Using columns: '{text_col}' for text and '{label_col}' for label.")

    print("Dataset loaded successfully.")
except Exception as e:
    print(f"Error loading dataset: {e}")
    df_sentiment = pd.DataFrame({'text': [], 'Sentiment': []})

if not df_sentiment.empty:
    # 2. Text Cleaning
    print("Cleaning text... (This might take a while for large datasets)")
    # Taking a subset if the dataset is too large for a quick run
    if len(df_sentiment) > 10000:
        df_sentiment = df_sentiment.sample(10000, random_state=42)

    df_sentiment['cleaned_text'] = df_sentiment[text_col].apply(lambda x: text_cleaning_pipeline(str(x), rule="lemmatize"))

    # 3. Train-Test Split
    X = df_sentiment['cleaned_text']
    y = df_sentiment[label_col]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # 4. TF-IDF Vectorization
    tfidf = TfidfVectorizer(max_features=5000)
    X_train_tfidf = tfidf.fit_transform(X_train)
    X_test_tfidf = tfidf.transform(X_test)

    # 5. Model Training and Evaluation
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_tfidf, y_train)

    y_pred = model.predict(X_test_tfidf)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))


Columns found: ['text', 'Sentiment']
Using columns: 'text' for text and 'Sentiment' for label.
Dataset loaded successfully.
Cleaning text... (This might take a while for large datasets)

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.96      0.87      1341
           1       0.85      0.50      0.63       659

    accuracy                           0.81      2000
   macro avg       0.82      0.73      0.75      2000
weighted avg       0.81      0.81      0.79      2000

